# **Quasi-experiments: difference-in-differences, synthetic control and regression discontinuity**

Practice for the module [“Causality and linear models”](https://ai-interpretability.school), lesson “Quasi-experiments”.

In the lesson the designs were formulas and tasks on tables of a few numbers. Here we compute them on real data and on a synthetic panel where the true effect is known. We check five claims of the lesson:

1. difference-in-differences rests on parallel trends, and their violation before the treatment starts is visible in the data;
2. synthetic control picks weights of the control units so that together they repeat the trajectory of the treated unit before the treatment;
3. synthetic DiD combines unit weights, time weights and difference-in-differences;
4. the two-way fixed effects regression (TWFE) is biased when groups are treated at different times and the effect grows over time, while the Callaway and Sant'Anna estimate is not;
5. regression discontinuity estimates the jump of the outcome at the threshold, and the answer depends on the width of the window.

Plan of the notebook:

1. the Proposition 99 data;
2. difference-in-differences;
3. synthetic control;
4. synthetic DiD;
5. difference-in-differences with variation in treatment timing;
6. regression discontinuity on US Senate elections;
7. where it breaks;
8. the same estimates in libraries.

The notebook runs on a CPU, the whole run takes about a minute.

In [ ]:
%pip install -q rdrobust==2.1.1 csdid==0.4.2 drdid==1.1.6 pysyncon==1.7.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. Data: Proposition 99

In 1988 California passed Proposition 99: the cigarette excise tax rose by 25 cents per pack, and the money went to anti-smoking programs. The law has been in force since 1989. The classic analysis is [Abadie, Diamond and Hainmueller, 2010](https://doi.org/10.1198/jasa.2009.ap08746); we take the table from the [synthdid](https://github.com/synth-inference/synthdid) package (BSD-3 license) on which [Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159) computed their estimates.

The table has 39 states for 1970–2000: California and 38 states without such programs or large tax increases in these years. The outcome `PacksPerCapita` is cigarette sales, packs per capita per year. There are 19 periods before the treatment (1970–1988) and 12 after (1989–2000).

In [ ]:
url = 'https://raw.githubusercontent.com/synth-inference/synthdid/master/data/california_prop99.csv'
prop99 = pd.read_csv(url, sep=';')
Y = prop99.pivot(index='State', columns='Year', values='PacksPerCapita')
treated = 'California'
controls = [s for s in Y.index if s != treated]
years = Y.columns.values
pre, post = years < 1989, years >= 1989
y_tr = Y.loc[treated].values          # California's trajectory
Y_co = Y.loc[controls].values         # 38 control states x 31 years
print(Y.shape, len(controls), pre.sum(), post.sum())

plt.figure(figsize=(8, 3.5))
plt.plot(years, y_tr, label='California')
plt.plot(years, Y_co.mean(0), label='mean of 38 states')
plt.axvline(1988.5, color='gray', ls='--')
plt.ylabel('packs per capita'); plt.legend(); plt.show()

Sales fall everywhere, but faster in California, and the gap grows even before 1989. Keep this in mind when reading the estimates below.

## 2. Difference-in-differences

$$\hat\tau_{DiD} = (\bar Y_{T,\text{after}} - \bar Y_{T,\text{before}}) - (\bar Y_{C,\text{after}} - \bar Y_{C,\text{before}}),$$

where $T$ is California, $C$ are the 38 control states, before is 1970–1988, after is 1989–2000.

**Task 1. Compute the difference-in-differences estimate for California: by the formula from means and as the coefficient of `treated:post` in the regression `PacksPerCapita ~ treated * post` over all 39 states and 31 years. Compare it with two naive estimates: the difference between California and the mean of the 38 states after 1989, and the difference between California after and before 1989. Round the answer to two decimal places.**

In [ ]:
# Your code here

Both naive estimates are larger in absolute value: the first mixes the effect with the fact that California smoked less before, the second with the general fall of sales in all states. Difference-in-differences removes both, if without the law the gap between California and the state average would have stayed constant. We check how well this holds in section 7.

## 3. Synthetic control

Instead of the mean of all 38 states, synthetic control builds a weighted mixture of states that repeats California's trajectory before 1989:

$$\hat w = \arg\min_{w \ge 0,\ \sum_j w_j = 1}\ \sum_{t \le 1988}\Big(Y_{\text{CA},t} - \sum_j w_j Y_{j,t}\Big)^2,\qquad \hat\tau_t = Y_{\text{CA},t} - \sum_j \hat w_j Y_{j,t}.$$

[Abadie, Diamond and Hainmueller, 2010](https://doi.org/10.1198/jasa.2009.ap08746) also fit predictors (income, cigarette price, the share of young people) with weights $V$; like [Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159), we fit only the past values of the outcome. This is a quadratic problem with constraints, solved by `cvxpy`. A small L2 penalty ($\zeta = 10^{-6}\hat\sigma$, as in the same paper) makes the solution unique.

In [ ]:
def sc_weights(y_target, Y_donors, zeta=1e-6):
    """Synthetic control weights: non-negative, summing to 1, fitted on the periods of y_target."""
    w = cp.Variable(Y_donors.shape[0])
    objective = cp.sum_squares(Y_donors.T @ w - y_target) + zeta ** 2 * len(y_target) * cp.sum_squares(w)
    cp.Problem(cp.Minimize(objective), [w >= 0, cp.sum(w) == 1]).solve(solver=cp.CLARABEL)
    return np.clip(w.value, 0, None)


w_sc = sc_weights(y_tr[pre], Y_co[:, pre])
weights = pd.Series(w_sc, index=controls).sort_values(ascending=False)
print(weights[weights > 0.01].round(3))
synthetic = w_sc @ Y_co
rmspe_sc = np.sqrt(((y_tr - synthetic)[pre] ** 2).mean())
level_did = Y_co.mean(0) + (y_tr[pre] - Y_co[:, pre].mean(0)).mean()   # the state mean shifted by the pre-1989 gap
rmspe_did = np.sqrt(((y_tr - level_did)[pre] ** 2).mean())
print(f'RMSPE before 1989: synthetic control {rmspe_sc:.2f} | mean of 38 states with a shift {rmspe_did:.2f}')

plt.figure(figsize=(8, 3.5))
plt.plot(years, y_tr, label='California')
plt.plot(years, synthetic, label='synthetic California')
plt.axvline(1988.5, color='gray', ls='--')
plt.ylabel('packs per capita'); plt.legend(); plt.show()

Six states got a nonzero weight, most of all Utah, Montana and Nevada. Before 1989 the synthetic California deviates from the real one by 1.66 packs on average, while the mean of the 38 states shifted by the level gap deviates by 7.16: the mixture repeats not only the level but also the slope.

**Task 2. Compute the synthetic control estimate of the effect: the mean gap between California and the synthetic California over 1989–2000. Round the answer to one decimal place.**

In [ ]:
# Your code here

In the paper by Arkhangelsky et al. the same estimate equals −19.6; the difference of 0.1 comes from the solver: synthetic control weights without regularization are not uniquely defined, as the authors note in footnote 5.

**Placebo across states.** Synthetic control with one treated unit has no standard errors. Abadie et al. propose to “assign” the law to each control state in turn, build a synthetic control for it, and compare the ratio of the error after 1989 to the error before. If California's ratio is the largest of 39, the probability of getting such a result by chance is $1/39 \approx 0.026$.

In [ ]:
ratio = {}
for s in Y.index:
    y_s, donors = Y.loc[s].values, Y.drop(index=s).values
    gap = y_s - sc_weights(y_s[pre], donors[:, pre]) @ donors
    ratio[s] = np.sqrt((gap[post] ** 2).mean()) / np.sqrt((gap[pre] ** 2).mean())
ratio = pd.Series(ratio).sort_values(ascending=False)
rank = list(ratio.index).index(treated) + 1
print(ratio.head(5).round(2))
print(f'California: place {rank} of {len(ratio)}, placebo p = {rank / len(ratio):.3f}')

In our specification, only past values of the outcome and no predictors, California is third of 39, and the placebo p equals 0.077. In Abadie et al. with predictors it is first. The conclusion about significance depends on the specification, while the effect estimate itself does not: in both cases it is about −19 packs.

## 4. Synthetic DiD

[Arkhangelsky et al., 2021](https://doi.org/10.1257/aer.20190159) combine two ideas. The state weights $\hat\omega$ are fitted as in synthetic control, but with a constant $\omega_0$, since California has to be matched in the shape of the trajectory, not in the level, and with an L2 penalty $\zeta^2 T_{pre}\|\omega\|^2$, where $\zeta = (N_{tr}T_{post})^{1/4}\hat\sigma$ and $\hat\sigma$ is the standard deviation of the yearly changes of the outcome in the control states before 1989. The year weights $\hat\lambda$ are fitted so that the weighted mean of the pre-treatment years of the control states repeats their mean after, also with a constant. The estimate is difference-in-differences with these weights:

$$\hat\tau_{SDID} = \Big(\bar Y_{\text{CA},post} - \sum_{t\le 1988}\hat\lambda_t Y_{\text{CA},t}\Big) - \sum_j \hat\omega_j\Big(\bar Y_{j,post} - \sum_{t\le 1988}\hat\lambda_t Y_{j,t}\Big).$$

The code below computes the weights; the estimate itself is task 3.

In [ ]:
T_pre, T_post, N_co = pre.sum(), post.sum(), len(controls)
sigma = np.diff(Y_co[:, pre], axis=1).std(ddof=1)
zeta = (1 * T_post) ** 0.25 * sigma

omega0, omega = cp.Variable(), cp.Variable(N_co)
cp.Problem(cp.Minimize(cp.sum_squares(omega0 + Y_co[:, pre].T @ omega - y_tr[pre]) + zeta ** 2 * T_pre * cp.sum_squares(omega)),
           [omega >= 0, cp.sum(omega) == 1]).solve(solver=cp.CLARABEL)
omega = np.clip(omega.value, 0, None)

lam0, lam = cp.Variable(), cp.Variable(T_pre)
cp.Problem(cp.Minimize(cp.sum_squares(lam0 + Y_co[:, pre] @ lam - Y_co[:, post].mean(1)) + (1e-6 * sigma) ** 2 * N_co * cp.sum_squares(lam)),
           [lam >= 0, cp.sum(lam) == 1]).solve(solver=cp.CLARABEL)
lam = np.clip(lam.value, 0, None)

print(f'sigma {sigma:.3f} | zeta {zeta:.3f}')
print(f'states with weight above 0.01: {(omega > 0.01).sum()} of {N_co}, largest weight {omega.max():.3f}')
print('year weights:', {int(y): float(round(v, 3)) for y, v in zip(years[pre], lam) if v > 0.001})

The penalty spreads the state weights: 22 states have weight above 0.01, none above 0.13. The year weights fell on 1986–1988: the “before” comparison uses years that resemble the post-treatment period.

**Task 3. Compute the synthetic DiD estimate by the formula above from the weights `omega` and `lam`. Round the answer to one decimal place.**

In [ ]:
# Your code here

Three estimates on the same data next to Table 1 of Arkhangelsky et al. (in the paper: DiD −27.3, synthetic control −19.6, SDID −15.6):

In [ ]:
did_m = (y_tr[post].mean() - y_tr[pre].mean()) - (Y_co[:, post].mean() - Y_co[:, pre].mean())
sc_m = (y_tr - w_sc @ Y_co)[post].mean()
sdid_m = (y_tr[post].mean() - lam @ y_tr[pre]) - omega @ (Y_co[:, post].mean(1) - Y_co[:, pre] @ lam)
pd.DataFrame({'our estimate': [did_m, sc_m, sdid_m], 'in the paper': [-27.3, -19.6, -15.6]},
             index=['DiD', 'synthetic control', 'synthetic DiD']).round(2)

## 5. Difference-in-differences with variation in treatment timing

Now a synthetic panel where the true effect is known. 300 units are observed over 10 periods: 100 are treated from period 4, 100 from period 7, 100 are never treated. The effect grows over time: in the first treated period it equals 1 and then adds 0.5 per period,

$$Y_{it} = a_i + 0.3\,t + \tau_{it} + \varepsilon_{it},\qquad \tau_{it} = \big(1 + 0.5\,(t - g_i)\big)\,\mathbf 1[t \ge g_i],$$

where $a_i$ is a constant difference between units (higher than in the never-treated group by 0.5 on average in the early group and by 0.2 in the late one), $g_i$ is the period when the treatment starts, $\varepsilon_{it} \sim \mathcal N(0, 0.5^2)$. Untreated trends are parallel by construction.

In [ ]:
def make_panel(slope, seed=SEED):
    rng = np.random.default_rng(seed)
    rows, uid = [], 0
    for g, n, shift in ((4, 100, 0.5), (7, 100, 0.2), (0, 100, 0.0)):   # g = 0: never treated
        for _ in range(n):
            a = rng.normal() + shift
            for t in range(1, 11):
                d = int(g > 0 and t >= g)
                tau = (1 + slope * (t - g)) if d else 0.0
                rows.append((uid, t, g, d, a + 0.3 * t + tau + rng.normal(0, 0.5), tau))
            uid += 1
    return pd.DataFrame(rows, columns=['id', 't', 'g', 'd', 'y', 'tau'])


panel = make_panel(slope=0.5)
true_att = panel.loc[panel.d == 1, 'tau'].mean()
print(f'observations {len(panel)} | true effect, mean over treated observations: {true_att:.3f}')

**Task 4. Estimate the effect by the regression with unit and period fixed effects (TWFE): `y ~ d + C(id) + C(t)`, where `d` indicates “the unit is already treated”. Compare the coefficient of `d` with the true effect. Round the answer to three decimal places.**

In [ ]:
# Your code here

The decomposition of [Goodman-Bacon, 2021](https://doi.org/10.1016/j.jeconom.2021.03.014) shows where the bias comes from: the TWFE coefficient is a weighted average of all “two groups, two periods” comparisons, and among them is the comparison of the late group with the early one, which by then is already treated. Let us compute it separately: periods 4–6 against 7–10, late group against early group.

In [ ]:
m = panel.groupby(['g', 't']).y.mean().unstack()
late_vs_early = (m.loc[7, 7:10].mean() - m.loc[7, 4:6].mean()) - (m.loc[4, 7:10].mean() - m.loc[4, 4:6].mean())
true_late = panel.loc[(panel.g == 7) & (panel.d == 1), 'tau'].mean()
print(f'late against the already treated early group: {late_vs_early:.3f} | true effect of the late group in periods 7-10: {true_late:.3f}')

The effect of the early group grew over these periods, and the comparison subtracts this growth from the effect of the late group: of the true 1.75 almost nothing is left.

[Callaway and Sant'Anna, 2021](https://doi.org/10.1016/j.jeconom.2020.12.001) forbid such comparisons. For each group $g$ and period $t \ge g$ they compute a separate effect as difference-in-differences with period $g-1$ and with the group that is never treated:

$$ATT(g, t) = E[Y_t - Y_{g-1} \mid G = g] - E[Y_t - Y_{g-1} \mid G = \infty],$$

and then average the effects $ATT(g,t)$ with weights proportional to group sizes.

**Task 5. Compute $ATT(g,t)$ for groups 4 and 7 and all periods $t \ge g$ from the means `m` and average them with weights by group size. Compare with the true effect and with TWFE. Round the answer to three decimal places.**

In [ ]:
# Your code here

## 6. Regression discontinuity: US Senate elections

The data of [Cattaneo, Frandsen and Titiunik, 2015](https://doi.org/10.1515/jci-2013-0010) come with the `rdrobust` package (GPL-3.0 license): 1390 US Senate elections, 1297 of them with both the margin and the next-election outcome recorded. The running variable `margin` is the Democratic candidate's margin at election $t$ in percentage points, the threshold is 0: with a margin above zero the Democrat wins. The outcome `vote` is the Democratic vote share in the next election for the same seat six years later. The jump at the threshold is the advantage of the party that already holds the seat (incumbency advantage), as in [Lee, 2008](https://doi.org/10.1016/j.jeconom.2007.05.004).

In [ ]:
from rdrobust import rdrobust
from rdrobust.datasets import rdrobust_RDsenate

senate = rdrobust_RDsenate().dropna(subset=['vote', 'margin'])
x, y = senate.margin.values, senate.vote.values
print(f'elections with both values: {len(senate)}')
print(f'naive difference of means right and left of the threshold: {y[x >= 0].mean() - y[x < 0].mean():.2f}')

rd = rdrobust(y=y, x=x)
h = rd.bws.loc['h', 'left']
print(f'rdrobust: estimate {rd.coef.loc["Conventional", "Coeff"]:.3f} | window h = {h:.3f}')

The naive difference is 21.70 points: right of the threshold are the states where Democrats are strong, left are those where they are weak. By default `rdrobust` fits a linear regression on each side of the threshold in a window $h$ that it chooses from an estimate of the mean squared error ([Calonico, Cattaneo and Titiunik, 2014](https://doi.org/10.3982/ECTA11757)), with a triangular kernel: the weight of a unit is $\max(0,\ 1 - |x|/h)$.

**Task 6. Reproduce the `rdrobust` estimate by hand: in the window $|x| \le h$ fit, on each side of the threshold, a weighted regression of `vote` on `margin` with weights $1 - |x|/h$ (`sm.WLS`) and take the difference of the intercepts. Round the answer to two decimal places.**

In [ ]:
# Your code here

## 7. Where it breaks

### 7.1. Parallel trends before the treatment

A check that uses no post-treatment data: pretend the law was passed in 1980 and compute difference-in-differences on 1970–1988 only. There was no effect in these years, and with parallel trends the estimate would be about zero.

In [ ]:
fake = years >= 1980
did_placebo = (y_tr[pre & fake].mean() - y_tr[pre & ~fake].mean()) - (Y_co[:, pre & fake].mean() - Y_co[:, pre & ~fake].mean())
print(f'placebo DiD with a fake start in 1980: {did_placebo:.2f}')

California cut sales faster than the others even before the law, and difference-in-differences attributes part of this trend to the law. Synthetic control and SDID choose the comparison by the pre-treatment trajectory, so their estimates are smaller in absolute value.

### 7.2. TWFE is biased only when the effect changes

The same panel, but with a constant effect (slope 0) and a faster growing one (slope 1):

In [ ]:
rows = []
for slope in (0.0, 0.5, 1.0):
    p = make_panel(slope)
    rows.append(dict(slope=slope, true=p.loc[p.d == 1, 'tau'].mean(),
                     TWFE=smf.ols('y ~ d + C(id) + C(t)', data=p).fit().params['d']))
pd.DataFrame(rows).round(3)

With a constant effect TWFE is almost unbiased: the comparison with the already treated group subtracts only its constant effect, which cancels. The faster the effect grows, the larger the bias.

### 7.3. The window of regression discontinuity

The estimate at different window widths, a plain linear regression on each side without a kernel:

In [ ]:
def rd_uniform(width):
    inside = np.abs(x) <= width
    fit = lambda mask: sm.OLS(y[mask], sm.add_constant(x[mask])).fit().params[0]
    return fit(inside & (x >= 0)) - fit(inside & (x < 0)), inside.sum()

pd.DataFrame([(w_, *rd_uniform(w_)) for w_ in (5, 10, 20, 40, 100)], columns=['h', 'jump', 'n']).round(3)

A narrow window (5 points, 245 elections) gives 9.8, a wide one (all 1297 elections) gives 6.0. A narrow window is closer to a comparison of “almost identical” elections but noisier; a wide one relies on linearity far from the threshold. This is why a regression discontinuity estimate is shown for several windows.

The second check is the density of the running variable at the threshold: if the margin could be manipulated, there would be more elections just above zero than just below. The McCrary test does this rigorously; here is a rough count.

In [ ]:
for b in (1, 2, 5):
    print(f'margin in [-{b}, 0): {((x >= -b) & (x < 0)).sum():4d} | in [0, {b}): {((x >= 0) & (x < b)).sum():4d}')

There is no jump in the number of elections at the threshold: 49 elections with a margin from −2 to 0 against 47 with a margin from 0 to 2.

## 8. The same estimates in libraries

Abadie et al.'s synthetic control is in the [`pysyncon`](https://github.com/sdfordham/pysyncon) package (MIT license). It fits both the state weights $W$ and the predictor weights $V$ by nested optimization; as predictors we pass the same 19 outcome values before 1989. The Callaway and Sant'Anna estimate is in the [`csdid`](https://github.com/d2cml-ai/csdid) package (MIT license).

In [ ]:
from pysyncon import Dataprep, Synth

years_pre = list(range(1970, 1989))
dataprep = Dataprep(foo=prop99, predictors=[], predictors_op='mean', time_predictors_prior=years_pre,
                    special_predictors=[('PacksPerCapita', [yr], 'mean') for yr in years_pre],
                    dependent='PacksPerCapita', unit_variable='State', time_variable='Year',
                    treatment_identifier=treated, controls_identifier=controls, time_optimize_ssr=years_pre)
synth = Synth()
synth.fit(dataprep=dataprep)
Z0, Z1 = dataprep.make_outcome_mats(time_period=range(1989, 2001))
print(synth.weights()[synth.weights() > 0.01].round(2))
print(f'pysyncon: {(Z1 - Z0 @ synth.W).mean():.1f} | our synthetic control: {(y_tr - w_sc @ Y_co)[post].mean():.1f}')

The same states with similar weights and an estimate of about −19. The number depends on the nested optimization of the weights $V$ and may differ slightly between runs and between machines.

In [ ]:
from csdid.att_gt import ATTgt

cs = ATTgt(yname='y', gname='g', idname='id', tname='t', data=panel).fit(est_method='reg', bstrap=False)
cs.aggte(typec='simple', bstrap=False)
print(f"csdid: {cs.atte['overall_att']:.3f}")

## What to take away

- **Difference-in-differences is checked before the treatment.** The placebo with a fake start in 1980 gives −11.51 packs: the trends are not parallel, and the estimate −27.35 is too large in absolute value.
- **Synthetic control picks the comparison by the trajectory.** Six states with weight repeat California before 1989 with an error of 1.66 packs against 7.16 for the state mean; the estimate is −19.5. The conclusion about significance from the placebo depends on the specification.
- **Synthetic DiD combines both ideas.** State weights with a constant and a penalty, year weights, difference-in-differences with weights: the estimate is −15.6, as in the paper.
- **With variation in treatment timing TWFE is biased if the effect changes over time.** 1.662 against the true 2.227; the culprit is the comparison with the group that is already treated. The Callaway and Sant'Anna estimate (2.297) makes no such comparisons.
- **Regression discontinuity estimates the effect at the threshold.** 7.41 points against the naive difference of means 21.70; the answer depends on the window, and it is shown for several windows.

Sources:
1. [Abadie, Diamond & Hainmueller, 2010. Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program](https://doi.org/10.1198/jasa.2009.ap08746)
2. [Arkhangelsky, Athey, Hirshberg, Imbens & Wager, 2021. Synthetic Difference-in-Differences](https://doi.org/10.1257/aer.20190159)
3. [Goodman-Bacon, 2021. Difference-in-differences with variation in treatment timing](https://doi.org/10.1016/j.jeconom.2021.03.014)
4. [Callaway & Sant'Anna, 2021. Difference-in-Differences with multiple time periods](https://doi.org/10.1016/j.jeconom.2020.12.001)
5. [Lee, 2008. Randomized experiments from non-random selection in U.S. House elections](https://doi.org/10.1016/j.jeconom.2007.05.004)
6. [Cattaneo, Frandsen & Titiunik, 2015. Randomization Inference in the Regression Discontinuity Design: An Application to Party Advantages in the U.S. Senate](https://doi.org/10.1515/jci-2013-0010)
7. [Calonico, Cattaneo & Titiunik, 2014. Robust Nonparametric Confidence Intervals for Regression-Discontinuity Designs](https://doi.org/10.3982/ECTA11757)